In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier
)
from win32comext.adsi.demos.scp import verbose
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings

warnings.filterwarnings("ignore")

In [2]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
    }

In [3]:
def compare_models(models: dict, X_train, X_test, y_train, y_test):
    print(f"{'模型':<30s} {'准确率':>8s} {'精确率':>8s} {'召回率':>8s} {'F1':>8s}")
    print("-" * 70)

    results = {}
    for name, model in models.items():
        metrics = evaluate_model(model, X_train, X_test, y_train, y_test)

        results[name] = metrics
        print(
            f"{name:<30s} "
            f"{metrics['accuracy']:>8.4f} "
            f"{metrics['precision']:>8.4f} "
            f"{metrics['recall']:>8.4f} "
            f"{metrics['f1']:>8.4f}"
        )

    return results

In [4]:
X, y = load_breast_cancer(return_X_y=True)
X

array([[1.799e+01, 1.038e+01, 1.228e+02, ..., 2.654e-01, 4.601e-01,
        1.189e-01],
       [2.057e+01, 1.777e+01, 1.329e+02, ..., 1.860e-01, 2.750e-01,
        8.902e-02],
       [1.969e+01, 2.125e+01, 1.300e+02, ..., 2.430e-01, 3.613e-01,
        8.758e-02],
       ...,
       [1.660e+01, 2.808e+01, 1.083e+02, ..., 1.418e-01, 2.218e-01,
        7.820e-02],
       [2.060e+01, 2.933e+01, 1.401e+02, ..., 2.650e-01, 4.087e-01,
        1.240e-01],
       [7.760e+00, 2.454e+01, 4.792e+01, ..., 0.000e+00, 2.871e-01,
        7.039e-02]], shape=(569, 30))

In [5]:
len(y)

569

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [7]:
models = {
    '决策树': DecisionTreeClassifier(random_state=42),
    '随机森林': RandomForestClassifier(n_estimators=100, random_state=42),
    'GBDT': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(
        n_estimators=100, eval_metric='logloss', random_state=42
    ),
    'LightGBM': LGBMClassifier(n_estimators=100, verbose=-1, random_state=42)
}

In [8]:
results = compare_models(models, X_train, X_test, y_train, y_test)

模型                                  准确率      精确率      召回率       F1
----------------------------------------------------------------------
决策树                              0.9415   0.9712   0.9352   0.9528
随机森林                             0.9708   0.9640   0.9907   0.9772
GBDT                             0.9591   0.9633   0.9722   0.9677
XGBoost                          0.9649   0.9811   0.9630   0.9720
LightGBM                         0.9474   0.9626   0.9537   0.9581


In [9]:
import pandas as pd

def compare_to_dataframe(models: dict, X_train, X_test, y_train, y_test):
    """返回 DataFrame 格式的对比结果"""
    rows = []
    for name, model in models.items():
        metrics = evaluate_model(model, X_train, X_test, y_train, y_test)
        metrics['model'] = name
        rows.append(metrics)
    df = pd.DataFrame(rows).set_index('model')
    return df.round(4)

df_results = compare_to_dataframe(models, X_train, X_test, y_train, y_test)
print(df_results)

          accuracy  precision  recall      f1
model                                        
决策树         0.9415     0.9712  0.9352  0.9528
随机森林        0.9708     0.9640  0.9907  0.9772
GBDT        0.9591     0.9633  0.9722  0.9677
XGBoost     0.9649     0.9811  0.9630  0.9720
LightGBM    0.9474     0.9626  0.9537  0.9581


In [10]:
df_results

,accuracy,precision,recall,f1
model,,,,
决策树,0.9415,0.9712,0.9352,0.9528
随机森林,0.9708,0.9640,0.9907,0.9772
GBDT,0.9591,0.9633,0.9722,0.9677
XGBoost,0.9649,0.9811,0.9630,0.9720
LightGBM,0.9474,0.9626,0.9537,0.9581
